# 04.4 · Encoding categories: one-hot, ordinal, target encoding

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/cloudops-one/training-ai-ml/blob/main/modules/04-trustworthy-models/04-encoding-categories/lesson.ipynb)

Read the [README](README.md) first. This notebook:

1. encodes a six-row table by hand three ways and checks it with scikit-learn,
2. shows a linear model being misled by label-encoded countries,
3. handles countries that appear only in the test set,
4. computes target encoding by hand, then shows how naive target encoding of a high-cardinality column leaks, and how `TargetEncoder` prevents it.

In [1]:
# --- data helper: works locally and on Colab -------------------------------
from pathlib import Path
import urllib.request

def data_path(name):
    """Return a local path to data/<name>, downloading from GitHub on Colab."""
    for parent in [Path.cwd(), *Path.cwd().parents]:
        candidate = parent / "data" / name
        if candidate.exists():
            return candidate
    url = f"https://raw.githubusercontent.com/cloudops-one/training-ai-ml/main/data/{name}"
    dest = Path(name)
    if not dest.exists():
        urllib.request.urlretrieve(url, dest)
    return dest

import sys, warnings
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
warnings.filterwarnings("ignore")
sys.path.insert(0, str(data_path("download.py").parent))
from download import load_retail

raw = load_retail()                                   # ~1 s after the first run
sales = raw[~raw["Invoice"].astype(str).str.startswith("C")]
sales = sales[(sales["Quantity"] > 0) & (sales["Price"] > 0)].dropna(subset=["Customer ID"]).drop_duplicates()
sales = sales.assign(Revenue=sales["Quantity"] * sales["Price"])

def customer_table(cutoff, months=3):
    """One row per customer: features from before `cutoff`, churned = no order in the next `months`."""
    cutoff = pd.Timestamp(cutoff)
    past = sales[sales["InvoiceDate"] < cutoff]
    future = sales[(sales["InvoiceDate"] >= cutoff) & (sales["InvoiceDate"] < cutoff + pd.DateOffset(months=months))]
    c = past.groupby("Customer ID").agg(orders=("Invoice", "nunique"), spend=("Revenue", "sum"),
                                        products=("StockCode", "nunique"), country=("Country", "first"),
                                        first_order=("InvoiceDate", "min"), last_order=("InvoiceDate", "max"))
    c["tenure_days"] = (cutoff - c["first_order"]).dt.days
    c["days_since_last"] = (cutoff - c["last_order"]).dt.days
    came_back = future.groupby("Customer ID")["Revenue"].sum().reindex(c.index).fillna(0) > 0
    c["churned"] = (~came_back).astype(int)
    return c.drop(columns=["first_order", "last_order"])

features = ["orders", "spend", "products", "tenure_days", "days_since_last"]
cust = customer_table("2011-09-01")
print(f"{len(cust):,} customers, churn rate {cust['churned'].mean():.1%}")

3,317 customers, churn rate 43.6%


## 1. Three encodings of the same table

In [2]:
six = pd.DataFrame({"customer": ["A", "B", "C", "D", "E", "F"],
                    "country":  ["UK", "DE", "FR", "UK", "FR", "DE"],
                    "plan":     ["S", "L", "M", "M", "S", "L"]})
six

,customer,country,plan
0,A,UK,S
1,B,DE,L
2,C,FR,M
3,D,UK,M
4,E,FR,S
5,F,DE,L


In [3]:
from sklearn.preprocessing import OrdinalEncoder, OneHotEncoder

label = OrdinalEncoder().fit_transform(six[["country"]])                       # alphabetical: DE=0, FR=1, UK=2
ordinal = OrdinalEncoder(categories=[["S", "M", "L"]]).fit_transform(six[["plan"]])   # our order: S=0, M=1, L=2
onehot = OneHotEncoder(sparse_output=False).fit(six[["country"]])

out = six.assign(country_label=label.ravel().astype(int), plan_ordinal=ordinal.ravel().astype(int))
out = out.join(pd.DataFrame(onehot.transform(six[["country"]]).astype(int),
                            columns=onehot.get_feature_names_out(), index=six.index))
out

,customer,country,plan,country_label,plan_ordinal,country_DE,country_FR,country_UK
0,A,UK,S,2,0,0,0,1
1,B,DE,L,0,2,1,0,0
2,C,FR,M,1,1,0,1,0
3,D,UK,M,2,1,0,0,1
4,E,FR,S,1,0,0,1,0
5,F,DE,L,0,2,1,0,0


The plan column has a real order (S < M < L), so 0, 1, 2 is honest. The country column has no order; 0, 1, 2 is just alphabetical, but a model will read it as "UK is twice as much as FR". One-hot gives each country its own column, with no order implied.

## 2. Label encoding misleads a linear model

The 20-customer table: predict `spend_next_month` from visits, items, recency and country.

In [4]:
small = pd.read_csv(data_path("shopwave_customers_small.csv"))
num = ["visits_last_month", "items_last_month", "days_since_last"]
print(small.groupby("country")["spend_next_month"].agg(["mean", "size"]).round(1))

          mean  size
country             
DE       167.8     7
FR       207.1     3
UK       139.6    10


In [5]:
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error

codes = small["country"].map({"DE": 0, "FR": 1, "UK": 2})           # alphabetical label encoding
X_label = small[num].assign(country_code=codes)
m_label = LinearRegression().fit(X_label, small["spend_next_month"])

X_onehot = pd.concat([small[num], pd.get_dummies(small["country"], drop_first=True, dtype=int)], axis=1)
m_onehot = LinearRegression().fit(X_onehot, small["spend_next_month"])

b = m_label.coef_[-1]
print(f"label-encoded: one coefficient {b:.2f} per code → DE +0, FR +{b:.2f}, UK +{2*b:.2f}")
print(f"one-hot:       FR +{m_onehot.coef_[-2]:.2f}, UK +{m_onehot.coef_[-1]:.2f} (relative to DE)")

label-encoded: one coefficient 3.58 per code → DE +0, FR +3.58, UK +7.16
one-hot:       FR +15.08, UK +7.32 (relative to DE)


In [6]:
for name, m, X in [("label", m_label, X_label), ("one-hot", m_onehot, X_onehot)]:
    print(f"{name:<8} training MAE £{mean_absolute_error(small['spend_next_month'], m.predict(X)):.2f}")

label    training MAE £7.46
one-hot  training MAE £6.35


With label encoding the model *must* rank the countries DE < FR < UK and space them equally, because one slope multiplies 0, 1, 2. The data says the opposite: once visits and items are accounted for, French customers spend the most. One-hot lets each country have its own number. (Training MAE is shown only to compare fits on the same 20 rows; there is no test set here.)

## 3. Real countries, and countries you have never seen

In [7]:
past = sales[sales["InvoiceDate"] < "2011-09-01"]
by_item = past.groupby(["Customer ID", "StockCode"])["Quantity"].sum().reset_index()
fav = by_item.sort_values("Quantity").groupby("Customer ID").tail(1).set_index("Customer ID")["StockCode"]
cust["fav_product"] = fav.reindex(cust.index)       # the product each customer bought most units of

from sklearn.model_selection import train_test_split
tr, te = train_test_split(cust, test_size=0.25, random_state=42, stratify=cust["churned"])
y_tr, y_te = tr["churned"], te["churned"]
base = ["days_since_last", "orders"]
print(f"{cust['country'].nunique()} countries, {cust['fav_product'].nunique()} different favourite products")

from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler

def fit_score(A, B):
    """Scale, fit a logistic regression on A (train), return (train acc, test acc)."""
    sc = StandardScaler().fit(A)
    m = LogisticRegression(max_iter=2000).fit(sc.transform(A), y_tr)
    return m.score(sc.transform(A), y_tr), m.score(sc.transform(B), y_te)

36 countries, 951 different favourite products


In [8]:
print("countries in test but not in train:", sorted(set(te["country"]) - set(tr["country"])))

countries in test but not in train: ['Brazil', 'Singapore', 'United Arab Emirates']


In [9]:
enc = OneHotEncoder(handle_unknown="ignore", sparse_output=False).fit(tr[["country"]])
rows = enc.transform(pd.DataFrame({"country": ["Brazil", "France"]}))
print(len(enc.get_feature_names_out()), "one-hot columns")
print("number of 1s in each row:  Brazil", int(rows[0].sum()), "  France", int(rows[1].sum()))
print("France's 1 is in column:", enc.get_feature_names_out()[rows[1].argmax()])

33 one-hot columns
number of 1s in each row:  Brazil 0   France 1
France's 1 is in column: country_France


Brazil was never in the training data, so `handle_unknown="ignore"` encodes it as all zeros: "none of the countries I know". Without that option, `transform` raises an error on the first new country, which is exactly what would happen at 9 a.m. on launch day.

Thirty-plus columns for countries with one or two customers each is wasteful. `min_frequency` groups the rare ones into a single "infrequent" column.

In [10]:
enc_grouped = OneHotEncoder(handle_unknown="infrequent_if_exist", min_frequency=20, sparse_output=False).fit(tr[["country"]])
print(enc_grouped.get_feature_names_out())

options = {
    "no country": (tr[base], te[base]),
    "label-encoded": (tr[base].assign(c=OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1).fit(tr[["country"]]).transform(tr[["country"]]).ravel()),
                      te[base].assign(c=OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1).fit(tr[["country"]]).transform(te[["country"]]).ravel())),
    "one-hot, grouped": (np.hstack([tr[base], enc_grouped.transform(tr[["country"]])]),
                         np.hstack([te[base], enc_grouped.transform(te[["country"]])])),
}
for name, (A, B) in options.items():
    a, t = fit_score(A, B)
    print(f"{name:<18} train {a:.1%}  test {t:.1%}")

['country_France' 'country_Germany' 'country_United Kingdom'
 'country_infrequent_sklearn']
no country         train 67.8%  test 66.7%
label-encoded      train 67.8%  test 66.7%
one-hot, grouped   train 67.8%  test 66.9%


For churn, country adds almost nothing: 90% of customers are in the UK. That is a useful result too. Encoding correctly matters most when the categories carry real signal and have no natural order.

## 4. Target encoding: by hand

In [11]:
ten = pd.DataFrame({"country": ["UK", "UK", "UK", "UK", "DE", "DE", "FR", "FR", "FR", "NL"],
                    "churned": [1, 0, 0, 1, 0, 0, 1, 0, 1, 1]})

global_mean = ten["churned"].mean()
stats = ten.groupby("country")["churned"].agg(["sum", "count", "mean"])
m = 2                                                     # smoothing strength
stats["smoothed"] = (stats["count"] * stats["mean"] + m * global_mean) / (stats["count"] + m)
print("global churn rate:", global_mean)
stats.round(3)

global churn rate: 0.5


,sum,count,mean,smoothed
country,,,,
DE,0,2,0.000,0.250
FR,2,3,0.667,0.600
NL,1,1,1.000,0.667
UK,2,4,0.500,0.500


Raw target encoding gives NL a perfect 1.0 from **one** customer, whose own label is now their feature. Smoothing pulls small groups towards the global rate (NL 1.0 → 0.667).

## 5. Target encoding leaks on high-cardinality columns

Each customer's favourite product has 951 different values: far too many to one-hot. Target encoding is the natural choice. Done naively (category means computed on the training rows and used on those same rows), it leaks each customer's own label into their feature.

In [12]:
means = y_tr.groupby(tr["fav_product"]).mean()
naive_tr = tr[base].assign(fav_te=tr["fav_product"].map(means))
naive_te = te[base].assign(fav_te=te["fav_product"].map(means).fillna(y_tr.mean()))

for name, (A, B) in {"without fav_product": (tr[base], te[base]),
                     "naive target encoding": (naive_tr, naive_te)}.items():
    a, t = fit_score(A, B)
    print(f"{name:<24} train {a:.1%}  test {t:.1%}")

without fav_product      train 67.8%  test 66.7%
naive target encoding    train 78.6%  test 62.0%


Training accuracy jumps by 11 points, test accuracy *drops* by 5. Many products are the favourite of only one customer, so on the training rows `fav_te` simply *is* the churn label, and the model learns to trust it. On new customers the trick does not work.

scikit-learn's `TargetEncoder` fixes this with **cross-fitting**: `fit_transform` splits the training rows into folds and encodes each fold using the means of the *other* folds, so no row ever sees its own label. It also applies smoothing automatically.

In [13]:
from sklearn.preprocessing import TargetEncoder

te_enc = TargetEncoder(random_state=0)
fav_tr = te_enc.fit_transform(tr[["fav_product"]], y_tr).ravel()     # cross-fitted on train
fav_te = te_enc.transform(te[["fav_product"]]).ravel()               # full train means, applied to test
a, t = fit_score(tr[base].assign(fav_te=fav_tr), te[base].assign(fav_te=fav_te))
print(f"TargetEncoder (cross-fitted)   train {a:.1%}  test {t:.1%}")

TargetEncoder (cross-fitted)   train 67.8%  test 66.6%


No more fake training accuracy, and the test score is back where it was. Here the favourite product carries little real signal, so the honest encoding adds nothing; the point is that the naive one made the model *worse* while making it look better.

Always use `fit_transform` on training data with `TargetEncoder`. Calling `fit(...)` then `transform(...)` on the same training rows skips the cross-fitting and brings the leak back.

## Quiz answers

1. Ordinal for `plan_size` (S < M < L has a real order); one-hot for `country` (no order).
2. A single coefficient multiplies the codes 0, 1, 2, so the model is forced to rank the categories in alphabetical order and space them equally. Here it made UK > FR, while the one-hot fit shows FR > UK.
3. It encodes an unseen category as all zeros instead of raising an error.
4. Each row's own label goes into its category's mean, so for small categories the feature contains the answer. Fix: smoothing plus cross-fitting (`TargetEncoder.fit_transform`).
5. $(3 \times 0.333 + 2 \times 0.5) / (3 + 2) = (1 + 1)/5 = 0.4$.

**Next:** [04.5 Scaling features](../05-scaling/)